DAP391m - Member A: Anomaly Detection Model Pipeline
Smart Home Energy Data (HomeC)
Step: Tạo nhãn -> Step 5 (Split) -> Step 6 (Model) -> Step 7 (Eval)
      -> Step 8 (Tuning) -> Step 9 (Save pipeline)

Cách dùng: chạy lần lượt từng SECTION. Trong Jupyter/Colab thì
tách mỗi SECTION thành 1 cell. Chạy file .py: python A_model_pipeline.py

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, confusion_matrix, roc_curve,
                             classification_report)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

DATA_PATH = "../data/HomeC_cleaned_final.csv"   # bản B đã làm sạch thêm
RANDOM_STATE = 42
DEV_MODE = False   # True = chạy nhanh trên mẫu 100k dòng khi dev; False = full 500k khi train cuối

SECTION 1 — LOAD + LÀM SẠCH CƠ BẢN
(B đã làm Data Understanding; ở đây ta chỉ chuẩn hoá lại cho chắc)

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw shape:", df.shape)

# 1.1 — File này là HomeC_cleaned_final.csv (output của B), đã có sẵn
# cột "datetime" (B tự convert), cột "time" (Unix) vẫn còn nhưng không dùng.
# Dataset của B cũng đã tự drop "House overall [kW]" / "Solar [kW]" (trùng cột)
# -> KHÔNG drop lại ở đây, nếu không sẽ lỗi KeyError.
df["datetime"] = pd.to_datetime(df["datetime"])

# 1.2 — Feature thời gian (phục vụ model + RQ), dùng cột "datetime" của B
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

if DEV_MODE:
    df = df.sample(n=100000, random_state=RANDOM_STATE).reset_index(drop=True)
    print("DEV_MODE: sampled to", df.shape)

Raw shape: (503910, 40)


SECTION 2 — TẠO NHÃN BẤT THƯỜNG ⭐ (việc quan trọng nhất)
Proxy label dựa trên ngưỡng thống kê mean + k*std của tổng tiêu thụ.
k=3 -> ~2.46% anomaly (đã verify trên dữ liệu thật, nằm trong 1-5%).
LƯU Ý TRUNG THỰC: đây là PROXY label, KHÔNG phải ground-truth thật.
Phải justify bằng baseline paper (B đã chuẩn bị) khi trình bày.

In [3]:
K = 3
mu = df["use [kW]"].mean()
sigma = df["use [kW]"].std()
threshold = mu + K * sigma
df["anomaly"] = (df["use [kW]"] > threshold).astype(int)

rate = df["anomaly"].mean() * 100
print(f"[Label] threshold={threshold:.3f} | anomaly rate={rate:.2f}%")
print(df["anomaly"].value_counts())
# Nếu rate <1% hoặc >5% -> chỉnh K (2.5 / 3.5) và ghi lý do vào báo cáo.

[Label] threshold=4.034 | anomaly rate=2.46%
anomaly
0    491492
1     12418
Name: count, dtype: int64


SECTION 3 — CHUẨN BỊ FEATURE / TARGET

In [4]:
# Bỏ các cột không dùng làm feature:
#  - time: đã tách ra hour/day/month
#  - use [kW]: CHÍNH là cột tạo ra nhãn -> phải BỎ, nếu giữ là leakage trắng trợn!
drop_cols = ["time", "datetime", "use [kW]", "anomaly"]

# One-hot 2 cột thời tiết dạng text (icon, summary) -> giữ tín hiệu cho RQ3
X = df.drop(columns=drop_cols)
X = pd.get_dummies(X, columns=["icon", "summary"], drop_first=True)
y = df["anomaly"]

print("Feature matrix:", X.shape)

Feature matrix: (503910, 60)


SECTION 4 — STEP 5: SPLIT + SMOTE (chỉ trên train)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit CHỈ trên train
X_test_s = scaler.transform(X_test)         # transform test (không fit -> tránh leakage)

sm = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = sm.fit_resample(X_train_s, y_train)  # SMOTE CHỈ trên train
print("After SMOTE:", X_train_bal.shape, "| pos rate:", y_train_bal.mean())

After SMOTE: (786388, 60) | pos rate: 0.5


SECTION 5 — STEP 6+7: TRAIN ≥5 MODEL + ĐÁNH GIÁ

In [6]:
def evaluate(name, y_true, y_pred, y_score=None):
    auc = roc_auc_score(y_true, y_score) if y_score is not None else np.nan
    return {
        "Model": name,
        "F1": round(f1_score(y_true, y_pred), 4),
        "AUC": round(auc, 4) if not np.isnan(auc) else None,
        "Precision": round(precision_score(y_true, y_pred), 4),
        "Recall": round(recall_score(y_true, y_pred), 4),
    }

results = []

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE),
    "LightGBM": LGBMClassifier(n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
}
# LƯU Ý TẢI: KHÔNG dùng SVC/One-Class SVM trên full 500k -> rất chậm/treo.

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    pred = model.predict(X_test_s)
    score = model.predict_proba(X_test_s)[:, 1]
    results.append(evaluate(name, y_test, pred, score))
    print(f"[done] {name}")

# Model thứ 5: IsolationForest (unsupervised, để so sánh 2 trường phái)
iso = IsolationForest(contamination=float(y_train.mean()), random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(X_train_s)  # unsupervised: không dùng nhãn
iso_pred = (iso.predict(X_test_s) == -1).astype(int)  # -1 = anomaly -> 1
iso_score = -iso.score_samples(X_test_s)  # điểm bất thường (cao = bất thường)
results.append(evaluate("IsolationForest", y_test, iso_pred, iso_score))
print("[done] IsolationForest")

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
print("\n===== BẢNG SO SÁNH MODEL (Step 7) =====")
print(results_df.to_string(index=False))

best_name = results_df.iloc[0]["Model"]
print("\nBest model:", best_name)

[done] LogisticRegression
[done] RandomForest
[done] XGBoost
[done] LightGBM
[done] IsolationForest

===== BẢNG SO SÁNH MODEL (Step 7) =====
             Model     F1    AUC  Precision  Recall
      RandomForest 0.9392 0.9996     0.9247  0.9541
           XGBoost 0.9312 0.9995     0.9152  0.9477
          LightGBM 0.8886 0.9989     0.8318  0.9537
LogisticRegression 0.3837 0.9706     0.2418  0.9283
   IsolationForest 0.0868 0.7708     0.0888  0.0849

Best model: RandomForest


SECTION 6 — STEP 8: TUNING model tốt nhất (ví dụ cho LightGBM/XGB/RF)
Chỉ tune 2-3 param cho kịp 3 ngày.

In [7]:
if best_name in ("LightGBM", "XGBoost", "RandomForest"):
    if best_name == "LightGBM":
        base = LGBMClassifier(n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        param_dist = {"n_estimators": [100, 200, 300],
                      "num_leaves": [31, 63, 127],
                      "learning_rate": [0.05, 0.1, 0.2]}
    elif best_name == "XGBoost":
        base = XGBClassifier(eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE)
        param_dist = {"n_estimators": [100, 200, 300],
                      "max_depth": [4, 6, 8],
                      "learning_rate": [0.05, 0.1, 0.2]}
    else:
        base = RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE)
        param_dist = {"n_estimators": [100, 200, 300],
                      "max_depth": [None, 10, 20],
                      "min_samples_split": [2, 5, 10]}

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(base, param_dist, n_iter=5, scoring="f1",
                                cv=cv, random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train_bal, y_train_bal)
    print("Best params:", search.best_params_)
    best_model = search.best_estimator_
else:
    best_model = models.get(best_name)

# Đánh giá lại sau tuning
final_pred = best_model.predict(X_test_s)
final_score = best_model.predict_proba(X_test_s)[:, 1]
print("\n===== SAU TUNING =====")
print(classification_report(y_test, final_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, final_pred))

Best params: {'n_estimators': 100, 'min_samples_split': 2, 'max_depth': None}

===== SAU TUNING =====
              precision    recall  f1-score   support

           0     0.9988    0.9980    0.9984     98298
           1     0.9247    0.9541    0.9392      2484

    accuracy                         0.9970    100782
   macro avg     0.9618    0.9761    0.9688    100782
weighted avg     0.9970    0.9970    0.9970    100782

Confusion matrix:
 [[98105   193]
 [  114  2370]]


SECTION 7 — STEP 9: BUILD PIPELINE CUỐI + SAVE
Pipeline gói scaler + model -> app chỉ cần load và gọi .predict()

In [8]:
final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", best_model),
])
# Fit lại pipeline trên toàn bộ train (chưa SMOTE) — hoặc dùng X_train_bal tuỳ chọn.
# Ở đây fit trên train gốc đã scale-free vì pipeline tự scale:
final_pipeline.fit(X_train, y_train)

joblib.dump(final_pipeline, "best_model.pkl")
joblib.dump(list(X.columns), "feature_columns.pkl")  # app cần đúng thứ tự cột
results_df.to_csv("model_comparison.csv", index=False)
print("\nSaved: best_model.pkl, feature_columns.pkl, model_comparison.csv")

# Hàm cho app gọi:
def predict_anomaly(input_dict):
    """input_dict: {feature_name: value}. Trả 0 (normal) / 1 (anomaly)."""
    cols = joblib.load("feature_columns.pkl")
    row = pd.DataFrame([input_dict]).reindex(columns=cols, fill_value=0)
    pipe = joblib.load("best_model.pkl")
    return int(pipe.predict(row)[0])


Saved: best_model.pkl, feature_columns.pkl, model_comparison.csv
